# Assignment 4: Retrieval-augmented generation

[Official instructions](https://liu-nlp.ai/dl4nlp/units/a2_2.html)

This notebook implements the LCEL chain option.
It compares retrieval-augmented and no-context answers from the same model on the same
PubMedQA questions. `QUICK_RUN=True` checks ten questions while keeping the full retrieval
corpus; use `False` for the full evaluation.


## Preliminaries


In [1]:
# Install the packages in this notebook's Python kernel; version bounds keep the APIs compatible.
# After the first installation, restart the kernel if Jupyter asks you to.
%pip install -q "torch>=2.6,<3" "transformers==4.57.1" "datasets>=3,<5" "accelerate>=1,<2" "nltk>=3.9,<4" "pandas>=2,<4" "matplotlib>=3.8,<4" "tqdm>=4.66" "langchain-core>=1,<2" "langchain-huggingface>=1,<2" "langchain-chroma>=1,<2" "langchain-text-splitters>=1,<2" "sentence-transformers>=5,<6" "certifi>=2024.2.2"



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import HuggingFace models plus LangChain components for retrieval, prompts and generation.
import gc
import json
import re
import time
import urllib.request
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import display, Markdown, FileLink
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, set_seed
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Use CUDA if available, then Apple MPS, then CPU; FORCE_CPU can override this choice.
def select_device(use_cpu=False):
    if use_cpu:
        return torch.device('cpu')
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


/Users/echaefs/Documents/Code/tdde09/dl4nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Choose the answer model, embedding model and retrieval settings.
# QUICK_RUN reduces the evaluated questions while retaining the full document corpus.
QUICK_RUN = False
FORCE_CPU = False
SEED = 101
# The language model writes answers; the separate embedding model only represents text for search.
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
CHUNK_SIZE, CHUNK_OVERLAP = 200, 30  # Measured in embedding-tokenizer tokens.
EMBEDDING_TOKEN_LIMIT = 256  # The selected MiniLM sentence model's sequence limit.
# Retrieve this many chunks per question; several chunks can originate from the same document.
TOP_K = 1
# Limit each generated answer to this many new tokens.
MAX_NEW_TOKENS = 64
QUICK_QUESTIONS = 10
DATA_DIR = Path('dl4nlp_data')
# This directory holds optional metric/example files; RAG does not train a model here.
OUTPUT_DIR = Path('outputs') / ('assignment4_quick' if QUICK_RUN else 'assignment4')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = select_device(FORCE_CPU)
set_seed(SEED)
pd.set_option('display.max_colwidth', 140)
# Overlap must be smaller than a chunk, and chunks must fit the embedding model.
assert 0 <= CHUNK_OVERLAP < CHUNK_SIZE < EMBEDDING_TOKEN_LIMIT
print('Device:', DEVICE, '| Mode:', 'ten-question check' if QUICK_RUN else 'full evaluation')


Device: mps | Mode: full evaluation


## Part 1 - Task 1.1: Loading and inspecting PubMedQA
Keep yes/no questions. The document corpus contains each abstract's context and long
answer, as specified in the assignment. Preserve document IDs and years in both tables.


In [4]:
# Download PubMedQA with verified HTTPS and build separate question and document tables.
import ssl
import certifi

# Add trusted CA certificates for Python installations without a configured CA bundle.
download_context = ssl.create_default_context()
download_context.load_verify_locations(cafile=certifi.where())

DATA_URL = 'https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json'
data_file = DATA_DIR / 'ori_pqal.json'
# Reuse the downloaded dataset on subsequent runs.
if not data_file.exists():
    with urllib.request.urlopen(DATA_URL, timeout=60, context=download_context) as response:
        data_file.write_bytes(response.read())
raw_data = json.loads(data_file.read_text())
data = pd.DataFrame.from_dict(raw_data, orient='index')
data.index = data.index.astype(str)
# Remove ambiguous maybe labels so the evaluation is binary yes/no.
data = data[data['final_decision'].isin(['yes', 'no'])].copy()
# The assignment defines each retrievable document as its contexts plus long answer.
documents = pd.DataFrame({
    'abstract': data.apply(lambda row: ' '.join(row['CONTEXTS'] + [row['LONG_ANSWER']]), axis=1),
    'year': data['YEAR'],
})
# Keep the question, reference label and source document ID for answer and retrieval evaluation.
questions = pd.DataFrame({
    'question': data['QUESTION'], 'year': data['YEAR'],
    'gold_label': data['final_decision'], 'gold_context': data['LONG_ANSWER'],
    'gold_document_id': documents.index,
})
# Ensure every question can be matched to its gold document by ID.
assert documents.index.is_unique
assert set(questions['gold_document_id']) == set(documents.index)
print('Documents:', len(documents), '| Questions:', len(questions))
display(questions.head(3))
display(documents.head(3))
print('Example abstract:')
print(documents.iloc[0]['abstract'])


Documents: 890 | Questions: 890


,question,year,gold_label,gold_context,gold_document_id
21645374,Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?,2011,yes,"Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle...",21645374
16418930,Landolt C and snellen e acuity: differences in strabismus amblyopia?,2006,no,"Using the charts described, there was only a slight overestimation of visual acuity by the Snellen E compared to the Landolt C, even in ...",16418930
9488747,"Syncope during bathing in infants, a pediatric form of water-induced urticaria?",1997,yes,"""Aquagenic maladies"" could be a pediatric form of the aquagenic urticaria.",9488747


,abstract,year
21645374,Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces pe...,2011
16418930,Assessment of visual acuity depends on the optotypes used for measurement. The ability to recognize different optotypes differs even if ...,2006
9488747,Apparent life-threatening events in infants are a difficult and frequent problem in pediatric practice. The prognosis is uncertain becau...,1997


Example abstract:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye MitoTra

## Step 2 - Task 2.1: Generative language model
The small instruction-tuned Qwen model is a practical starting point. Use its native
chat template and return only newly generated text. Greedy generation keeps the
RAG/no-context comparison deterministic for a fixed model and runtime.


In [5]:
# Wrap a pretrained instruction model in a LangChain-compatible text-generation pipeline.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if not tokenizer.chat_template:
    raise ValueError('Choose an instruction model with a chat template.')
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
# Use BF16/FP16 on suitable CUDA hardware and float32 on CPU/MPS.
inference_dtype = (torch.bfloat16 if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
                   else torch.float16 if DEVICE.type == 'cuda' else torch.float32)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=inference_dtype)
# Greedy decoding (do_sample=False) chooses the most likely next token at each step.
text_generator = pipeline(
    'text-generation', model=model, tokenizer=tokenizer, device=DEVICE,
    max_new_tokens=MAX_NEW_TOKENS, do_sample=False, return_full_text=False,
    pad_token_id=tokenizer.pad_token_id,
)
# Adapt the HuggingFace pipeline so it can be called from LangChain using invoke().
language_model = HuggingFacePipeline(pipeline=text_generator)
# Use the model's chat template to insert its expected role markers and assistant prefix.
sanity_prompt = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': 'Is Stockholm the capital of Sweden? Answer Yes or No.'}],
    tokenize=False, add_generation_prompt=True,
)
print(language_model.invoke(sanity_prompt))


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use mps
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Yes. Stockholm is indeed the capital and largest city of Sweden. It serves as the administrative center for the country and is known for its rich history, culture, and modern infrastructure.


## Part 3: Document database
### Task 3.1: Embedding model
Embed questions and document chunks in the same vector space. Normalized embeddings
and cosine distance are used for retrieval.


In [6]:
# Use a sentence-embedding model to represent questions and passages in a shared vector space.
# Normalize embeddings to unit length so cosine similarity compares their directions.
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME, model_kwargs={'device': str(DEVICE)},
    encode_kwargs={'normalize_embeddings': True},
)
# Count chunk tokens with this model's tokenizer, which may differ from the generator's tokenizer.
embedding_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
# Check that one question becomes one numerical vector of the expected form.
probe_embedding = np.asarray(embedding_model.embed_query('What is programmed cell death?'))
assert probe_embedding.ndim == 1
print('Embedding shape:', probe_embedding.shape)
print('Embedding norm:', np.linalg.norm(probe_embedding))


Embedding shape: (384,)
Embedding norm: 1.000000002389936


### Task 3.2: Chunking
Chunk length is measured with the embedding model's tokenizer. The default 200-token
size leaves room below the selected model's 256-token limit. Overlap can preserve
evidence crossing chunk boundaries, while excessive overlap duplicates retrieved content.


In [7]:
# Split long documents into overlapping passages that fit the embedding model's token limit.
def count_embedding_tokens(text):
    # Count text tokens without special markers when measuring candidate chunk lengths.
    return len(embedding_tokenizer.encode(text, add_special_tokens=False))

# Prefer natural text boundaries while keeping token counts within the configured chunk size.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
    length_function=count_embedding_tokens,
)
# Carry the original document ID into every chunk so retrieval can be checked against the gold ID.
metadata = [{'id': str(index), 'year': str(row['year'])} for index, row in documents.iterrows()]
source_documents = splitter.create_documents(documents['abstract'].tolist(), metadatas=metadata)
# Split the prepared Document objects; splitting already-short passages leaves them unchanged.
chunks = splitter.split_documents(source_documents)
# Check the final lengths including special tokens, preventing silent embedding truncation.
token_lengths = [len(embedding_tokenizer.encode(chunk.page_content, add_special_tokens=True)) for chunk in chunks]
assert max(token_lengths) <= EMBEDDING_TOKEN_LIMIT, 'Reduce the chunk size to prevent embedding truncation.'
print('Documents:', len(source_documents), '| Chunks:', len(chunks), '| Maximum chunk tokens:', max(token_lengths))
display(pd.DataFrame([{'document_id': chunk.metadata['id'], 'tokens': size, 'text': chunk.page_content}
                      for chunk, size in list(zip(chunks, token_lengths))[:5]]))


Documents: 2170 | Chunks: 2170 | Maximum chunk tokens: 202


,document_id,tokens,text
0,21645374,202,Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces pe...
1,21645374,202,"cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves..."
2,21645374,157,"a significantly lower number of perforations compared to controls, and that displayed mitochondrial dynamics similar to that of non-PCD ..."
3,16418930,202,Assessment of visual acuity depends on the optotypes used for measurement. The ability to recognize different optotypes differs even if ...
4,16418930,202,"per line had to be correctly identified, while wrong answers were monitored. In the group of patients, the eyes with the lower visual ac..."


### Task 3.3: Vector store
Chroma stores the chunks and embeddings. The collection is recreated when this cell
runs so repeated notebook execution does not duplicate entries. Smaller cosine
distances indicate closer matches.


In [8]:
# Index passage embeddings in Chroma and test retrieval with one example question.
vector_store = Chroma(collection_name='dl4nlp_assignment4', embedding_function=embedding_model,
                      collection_metadata={'hnsw:space': 'cosine'})
# Recreate the collection on rerun to avoid adding duplicate copies of each passage.
vector_store.reset_collection()
# Add passages in manageable batches; Chroma obtains embeddings through embedding_model.
for offset in range(0, len(chunks), 256):
    batch_chunks = chunks[offset:offset + 256]
    vector_store.add_documents(batch_chunks, ids=[f'chunk_{offset + i}' for i in range(len(batch_chunks))])
# Returned scores are cosine distances, so smaller values mean closer matches.
sanity_matches = vector_store.similarity_search_with_score('What is programmed cell death?', k=3)
display(pd.DataFrame([{'document_id': doc.metadata['id'], 'cosine_distance': score, 'text': doc.page_content}
                      for doc, score in sanity_matches]))


,document_id,cosine_distance,text
0,21645374,0.523371,Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces pe...
1,15223779,0.674422,"in cultured uveal melanoma cells compared with cutaneous melanoma cells. Moreover, in contrast to cutaneous melanoma cell lines c-kit ma..."
2,18603989,0.680941,Embalming is the through disinfection and art of preserving bodies after death using chemical substances. It keeps a body life like in a...


## Part 4 - Task 4.1: Full RAG pipeline (Option B: LCEL)
A parallel runnable returns the retrieved documents and the original question.
The answer chain formats the evidence and question, applies the native chat template,
and generates an answer. The baseline uses the same chain with an empty context.


In [9]:
# Build two chains: retrieved evidence plus generation, and generation with an empty context.
def build_chains(llm, tokenizer, retriever):
    # Request a leading Yes/No label so answers can be evaluated automatically.
    template = ChatPromptTemplate.from_messages([
        ('system', 'Answer the research question with Yes or No, followed by one short explanation. '
                   'Use the supplied context when it is available.'),
        ('human', 'Context:\n{context}\n\nQuestion: {question}'),
    ])

    # Join the retrieved passage texts into the context string used by the prompt.
    def prompt_inputs(item):
        context = '\n\n'.join(doc.page_content for doc in item['context'])
        return {'question': item['question'], 'context': context or '(No context provided.)'}

    def render_chat(prompt_value):
        # Translate LangChain role names to the roles expected by the HuggingFace chat template.
        role_map = {'system': 'system', 'human': 'user', 'ai': 'assistant'}
        messages = [{'role': role_map[message.type], 'content': message.content}
                    for message in prompt_value.to_messages()]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # The | operator passes each stage's output to the next: format -> prompt -> chat text -> model -> string.
    answer_chain = (RunnableLambda(prompt_inputs) | template | RunnableLambda(render_chat)
                    | llm | StrOutputParser())
    # Prepare both retrieved context and the unchanged question; assign adds the generated answer.
    rag = RunnableParallel(context=retriever, question=RunnablePassthrough()).assign(answer=answer_chain)
    # Use the same answer chain with an empty context to measure the effect of retrieval.
    baseline = RunnableParallel(context=RunnableLambda(lambda _: []),
                                question=RunnablePassthrough()).assign(answer=answer_chain)
    return rag, baseline


In [10]:
# Run one complete RAG example and inspect both its answer and the retrieved document IDs.
retriever = vector_store.as_retriever(search_kwargs={'k': TOP_K})
rag_chain, baseline_chain = build_chains(language_model, tokenizer, retriever)
# The result contains the original question, retrieved Document objects, and answer text.
example_result = rag_chain.invoke(questions.iloc[0]['question'])
print('Question:', questions.iloc[0]['question'])
print('Answer:', example_result['answer'])
print('Retrieved document IDs:', [doc.metadata['id'] for doc in example_result['context']])


Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Answer: Yes.
Retrieved document IDs: ['21645374']


## Part 5: Evaluation
### Task 5.1: Answer accuracy and no-context baseline
Accept a leading Yes or No label. Report the valid-answer fraction, accuracy and F1
among valid answers, and accuracy counting invalid answers as incorrect. This avoids
hiding failures to follow the requested output format.


In [11]:
# Convert model text to yes/no labels and measure answer validity, accuracy and F1.
def parse_label(text):
    # Accept a leading label only; do not search the explanation for a convenient yes/no.
    match = re.match(r'^\s*(?:answer\s*:\s*)?(yes|no)\b', text, re.IGNORECASE)
    # Return None when the answer does not begin with an accepted label.
    return match.group(1).lower() if match else None


def classification_metrics(gold, predicted):
    if len(gold) != len(predicted) or not gold:
        raise ValueError('Gold and predicted labels must have equal, nonzero lengths.')
    # Keep recognizable yes/no predictions for valid-only metrics; report their fraction separately.
    valid = [(g, p) for g, p in zip(gold, predicted) if p in ('yes', 'no')]
    correct = sum(g == p for g, p in valid)
    # Treat yes as the positive class: TP = correct yes, FP = wrong yes, FN = missed yes.
    tp = sum(g == p == 'yes' for g, p in valid)
    fp = sum(g == 'no' and p == 'yes' for g, p in valid)
    fn = sum(g == 'yes' and p == 'no' for g, p in valid)
    # F1 = 2*TP / (2*TP + FP + FN); a zero denominator gives 0 by convention here.
    denominator = 2 * tp + fp + fn
    # All-question accuracy counts invalid answers as wrong; valid-only accuracy excludes them.
    return {'questions': len(gold), 'valid_answers': len(valid),
            'valid_answer_fraction': len(valid) / len(gold),
            'accuracy_all_invalid_counted_wrong': correct / len(gold),
            'accuracy_valid_only': correct / len(valid) if valid else None,
            'f1_yes_valid_only': (2 * tp / denominator if denominator else 0.0) if valid else None}


In [12]:
# Evaluate RAG and the no-context baseline on exactly the same questions.
evaluation_questions = questions.iloc[:QUICK_QUESTIONS] if QUICK_RUN else questions
run_config = {
    'quick_run': QUICK_RUN, 'model': MODEL_NAME, 'embedding_model': EMBEDDING_MODEL_NAME,
    'seed': SEED, 'top_k_chunks': TOP_K, 'chunk_size_tokens': CHUNK_SIZE,
    'chunk_overlap_tokens': CHUNK_OVERLAP, 'max_new_tokens': MAX_NEW_TOKENS,
    'documents': len(documents), 'chunks': len(chunks), 'questions': len(evaluation_questions),
}
# Optional record of the evaluation settings.
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2))
records = []
start = time.perf_counter()
# Optional logging: keep one JSON record per question as the evaluation progresses.
with (OUTPUT_DIR / 'predictions.jsonl').open('w', encoding='utf-8') as stream:
    for _, question in tqdm(evaluation_questions.iterrows(), total=len(evaluation_questions)):
        # Answer each question once with retrieved evidence and once with no evidence.
        rag_output = rag_chain.invoke(question['question'])
        baseline_output = baseline_chain.invoke(question['question'])
        # Deduplicate document IDs because multiple retrieved chunks can share a source document.
        retrieved_ids = list(dict.fromkeys(str(doc.metadata['id']) for doc in rag_output['context']))
        # Retain answers and source passages so aggregate errors can be inspected later.
        record = {
            'question': question['question'], 'gold_label': question['gold_label'],
            'gold_document_id': str(question['gold_document_id']), 'year': str(question['year']),
            'gold_context': question['gold_context'],
            'rag_answer': rag_output['answer'], 'baseline_answer': baseline_output['answer'],
            'rag_label': parse_label(rag_output['answer']), 'baseline_label': parse_label(baseline_output['answer']),
            'retrieved_document_ids': retrieved_ids,
            'gold_document_retrieved': str(question['gold_document_id']) in retrieved_ids,
            'retrieved_chunks': [{'text': doc.page_content, 'metadata': doc.metadata} for doc in rag_output['context']],
        }
        records.append(record)
        stream.write(json.dumps(record, ensure_ascii=False) + chr(10))
        # Write each completed result promptly so progress is available if a long run is interrupted.
        stream.flush()
evaluation_seconds = time.perf_counter() - start
results = pd.DataFrame(records)
gold = results['gold_label'].tolist()
# Compute both setups' scores using exactly the same metric definitions.
rag_metrics = classification_metrics(gold, results['rag_label'].tolist())
baseline_metrics = classification_metrics(gold, results['baseline_label'].tolist())
metrics_table = pd.DataFrame([{'setup': 'RAG', **rag_metrics}, {'setup': 'No context', **baseline_metrics}])
display(metrics_table)
print('RAG minus no-context accuracy (invalid answers counted wrong):',
      rag_metrics['accuracy_all_invalid_counted_wrong'] - baseline_metrics['accuracy_all_invalid_counted_wrong'])


100%|██████████| 890/890 [28:57<00:00,  1.95s/it] 


,setup,questions,valid_answers,valid_answer_fraction,accuracy_all_invalid_counted_wrong,accuracy_valid_only,f1_yes_valid_only
0,RAG,890,890,1.0,0.670787,0.670787,0.774094
1,No context,890,890,1.0,0.620225,0.620225,0.765603


RAG minus no-context accuracy (invalid answers counted wrong): 0.05056179775280889


### Task 5.2: Retrieval quality and detailed inspection
A retrieval hit means at least one of the top-k chunks comes from the gold document.
Inspect both retrieval misses and generation failures after successful retrieval.


In [13]:
# Measure whether retrieval found the source document and inspect retrieval/generation errors separately.
# A hit means at least one retrieved chunk comes from the question's gold document.
retrieval_hit_rate = float(results['gold_document_retrieved'].mean())
print(f'Gold-document hit rate among top {TOP_K} chunk(s): {retrieval_hit_rate:.3f}')
metrics = {
    'rag': rag_metrics, 'no_context': baseline_metrics,
    'gold_document_hit_rate_at_k_chunks': retrieval_hit_rate,
    'k_chunks': TOP_K, 'evaluated_questions': len(records), 'evaluation_seconds': evaluation_seconds,
}
# Optional file copy of the aggregate measurements.
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
# Separate correct answers, failures despite retrieving the gold source, and retrieval misses.
groups = {
    'Correct answer with gold document retrieved': [r for r in records if r['gold_document_retrieved'] and r['rag_label'] == r['gold_label']],
    'Wrong or invalid answer despite gold document': [r for r in records if r['gold_document_retrieved'] and r['rag_label'] != r['gold_label']],
    'Gold document missed': [r for r in records if not r['gold_document_retrieved']],
}
# Choose up to two examples per category for detailed reading.
inspected = {name: examples[:2] for name, examples in groups.items()}
# Optional file copy of these inspection examples.
(OUTPUT_DIR / 'inspection_examples.json').write_text(json.dumps(inspected, indent=2))
for name, examples in inspected.items():
    display(Markdown('#### ' + name))
    if not examples:
        print('No examples in this category in the evaluated subset.')
    for example in examples:
        print('Question:', example['question'])
        print('Gold label:', example['gold_label'], '| RAG answer:', example['rag_answer'])
        print('Gold document:', example['gold_document_id'], '| Retrieved:', example['retrieved_document_ids'])
        # Read the retrieved passage: a gold-document hit alone does not guarantee the useful sentence was fetched.
        for chunk in example['retrieved_chunks']:
            print('Retrieved context:', chunk['text'])
        print()


Gold-document hit rate among top 1 chunk(s): 0.972


#### Correct answer with gold document retrieved

Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold label: yes | RAG answer: Yes.
Gold document: 21645374 | Retrieved: ['21645374']
Retrieved context: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PC

#### Wrong or invalid answer despite gold document

Question: Landolt C and snellen e acuity: differences in strabismus amblyopia?
Gold label: no | RAG answer: Yes.
Gold document: 16418930 | Retrieved: ['16418930']
Retrieved context: the charts described, there was only a slight overestimation of visual acuity by the Snellen E compared to the Landolt C, even in strabismus amblyopia. Small differences in the lower visual acuity range have to be considered.

Question: Can tailored interventions increase mammography use among HMO women?
Gold label: yes | RAG answer: No.
Gold document: 10808977 | Retrieved: ['10808977']
Retrieved context: answered about why the tailored print intervention was not more powerful. Nevertheless, it is clear that additional interventions will be needed to maintain women's adherence to mammography. Medical Subject Headings (MeSH): mammography screening, telephone counseling, tailored print communications, barriers.



#### Gold document missed

Question: Double balloon enteroscopy: is it efficacious and safe in a community setting?
Gold label: yes | RAG answer: Yes, double balloon enteroscopy (DBE) appears to be efficacious and safe in a community setting for managing severe acute obscure gastrointestinal bleeding (OGIB), especially in cases where antegrade and retrograde DBE have been unsuccessful. This method allows for immediate visualization of the bleeding site without the need for contrast
Gold document: 23831910 | Retrieved: ['25251991']
Retrieved context: There are few data concerning emergency double-balloon enteroscopy (DBE) and its usefulness in the management of severe acute obscure gastrointestinal bleeding (OGIB). The aim of this retrospective study was to evaluate emergency DBE and capsule endoscopy (CE) in patients with overt OGIB, analyzing the feasibility of this combined approach. Emergency DBE in patients with overt OGIB was defined as performance within 24 h of symptom onset. We reported 27 patients (16 m

### Observations

Across 890 questions, RAG achieved 67.08% accuracy, compared with 62.02% without retrieved context, an improvement of 5.06 percentage points. Both approaches produced recognizable leading yes/no labels for every question. F1 for the “yes” class increased slightly, from 0.7656 to 0.7741.

The no-context baseline answered “yes” to all 890 questions. Its accuracy therefore exactly matched the proportion of “yes” labels in the dataset, rather than demonstrating an ability to distinguish the two classes. RAG correctly changed 95 baseline errors to “no”, but also changed 50 previously correct answers to an incorrect “no”. This produced a net improvement of 45 correct answers.

Retrieval returned a chunk from the gold document for 865 questions, giving a document hit rate of 97.19%. However, 280 answers were still wrong despite retrieving the gold document. Retrieving the correct document does not necessarily retrieve the specific evidence needed. For example, the mammography question received a chunk containing the end of the conclusion but missing the main positive finding about telephone counselling, and the model incorrectly answered “no”.

Larger chunks or retrieving more than one chunk might provide missing evidence, but this would require another experiment to verify. These results measure retrieval and answer generation within the supplied corpus, which includes the source abstracts and conclusions.